In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC18__NearMiss_SMOTE__v1__pca4_knn__v1"
CARPETA_DATASET = "CIC18__NearMiss_SMOTE__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KNN =====
N_NEIGHBORS = 5
WEIGHTS = "distance"      # "uniform" o "distance"
METRIC = "minkowski"      # euclidean suele ser minkowski con p=2
P = 2

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC18__NearMiss_SMOTE__v1/CIC18__NearMiss_SMOTE__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC18__NearMiss_SMOTE__v1/CIC18__NearMiss_SMOTE__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(150000, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,2235,0,527,12,1,3,1,3,0,3,...,34,1,0,0,0,0,0,0,0,0
1,2464,0,7953,12,1,3,1,3,0,3,...,34,1,0,0,0,0,0,0,0,0
2,2541,0,7953,12,1,3,1,3,0,3,...,34,1,0,0,0,0,0,0,0,0
3,2310,0,591,12,1,3,1,3,0,3,...,85,1,0,0,0,0,0,0,0,0
4,2513,0,527,12,1,3,1,3,0,3,...,57,1,0,0,0,0,0,0,0,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,10000


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (150000, 54)
Shape y_train: (150000,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("knn", KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=WEIGHTS,
        metric=METRIC,
        p=P
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef),

    "roc_auc": "roc_auc_ovr_weighted",
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc', 'test_roc_auc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "roc_auc": cv_results["test_roc_auc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.983567,0.983584,0.983567,0.983541,0.983584,0.983567,0.983541,0.982398,0.997176,0.175480,0.223314
1,2,0.983167,0.983164,0.983167,0.983145,0.983164,0.983167,0.983145,0.981967,0.997111,0.235948,0.535204
2,3,0.982033,0.982014,0.982033,0.982004,0.982014,0.982033,0.982004,0.980753,0.996543,0.176565,0.227829
3,4,0.982867,0.982833,0.982867,0.982824,0.982833,0.982867,0.982824,0.981647,0.996819,0.208955,0.243308
4,5,0.983133,0.983140,0.983133,0.983089,0.983140,0.983133,0.983089,0.981936,0.997188,0.234133,0.266311


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC18__NearMiss_SMOTE__v1__pca4_knn__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC18__NearMiss_SMOTE__v1/CIC18__NearMiss_SMOTE__v1__train.csv',
 'shape_train': {'rows': 150000, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_media': {'accuracy': 0.9829533333333333,
  'precision_weighted': 0.9829470805148863,
  'recall_weighted': 0.9829533333333333,
  'f1_weighted': 0.9829205981527336,
  'precision_macro': 0.9829470805148863,
  'recall_macro': 0.9829533333333333,
  'f1_macro': 0.9829205981527336,
  'mcc': 0.981740108031443,
  'roc_auc': 0.9969674888095239,
  'fit_time': 0.20621604919433595,
  'score_time': 0.2991933345794678},
 'metricas_std': {'accuracy': 0.0005718391382198433,
  'precision_weighted': 0.0005860623240478458,
  'recall_weighted': 0.0005718391

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print(f"ROC AUC              : {summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.982953 ± 0.000572

Precision weighted  : 0.982947 ± 0.000586
Recall weighted     : 0.982953 ± 0.000572
F1 weighted         : 0.982921 ± 0.000573

Precision macro     : 0.982947 ± 0.000586
Recall macro        : 0.982953 ± 0.000572
F1 macro            : 0.982921 ± 0.000573

MCC                 : 0.981740 ± 0.000614
ROC AUC              : 0.996967 ± 0.000280

Fit time medio      : 0.206216
Score time medio    : 0.299193


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.983567,0.983584,0.983567,0.983541,0.983584,0.983567,0.983541,0.982398,0.997176,0.175480,0.223314
1,2,0.983167,0.983164,0.983167,0.983145,0.983164,0.983167,0.983145,0.981967,0.997111,0.235948,0.535204
2,3,0.982033,0.982014,0.982033,0.982004,0.982014,0.982033,0.982004,0.980753,0.996543,0.176565,0.227829
3,4,0.982867,0.982833,0.982867,0.982824,0.982833,0.982867,0.982824,0.981647,0.996819,0.208955,0.243308
4,5,0.983133,0.983140,0.983133,0.983089,0.983140,0.983133,0.983089,0.981936,0.997188,0.234133,0.266311


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(335288, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,4790,0,1343807,1,3,163,1,6,10,180,...,3,3,0,0,0,0,0,0,0,0
1,2,0,13905,3,13,371,1459,261,0,443,...,156,3,0,0,0,0,0,0,0,2
2,2,0,5725,3,13,2844,1459,137,0,4649,...,156,3,0,0,0,0,0,0,0,2
3,2,0,217626,3,13,230,1459,203,0,20004,...,156,3,0,0,0,0,0,0,0,2
4,78205,5,456956,756,1066,16372,72605,1778,443,134586,...,4725,278,72650,56255,70259,43755,32542,11323,36885,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,90000
1,90000
2,39772
3,29040
4,28907
5,27955
6,18810
7,8281
8,1982


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (335288, 54)
Shape y_test: (335288,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 335288


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.40879482713368803,
 'precision_weighted': 0.7915511516878581,
 'recall_weighted': 0.40879482713368803,
 'f1_weighted': 0.4799704398015179,
 'precision_macro': 0.500254392175916,
 'recall_macro': 0.6778213100727263,
 'f1_macro': 0.4145650433710809,
 'roc_auc': 0.7171753331657125,
 'mcc': 0.3886948959213576}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")

print(f"ROC AUC              : {metricas_test['roc_auc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.408795

Precision weighted  : 0.791551
Recall weighted     : 0.408795
F1 weighted         : 0.479970

Precision macro     : 0.500254
Recall macro        : 0.677821
F1 macro            : 0.414565

MCC                 : 0.388695
ROC AUC              : 0.717175


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,20477,1774,874,4530,250,6413,518,6104,10206,1072,18151,1981,17648,0,2
1,0,32709,604,0,0,0,0,10547,3455,541,29846,105,12193,0,0
2,0,564,6219,0,0,0,6261,5642,0,0,138,3,20945,0,0
3,0,0,0,29030,0,10,0,0,0,0,0,0,0,0,0
4,7100,0,46,0,8626,0,48,0,11,0,0,335,12740,0,1
5,0,0,0,9333,0,15886,0,0,2736,0,0,0,0,0,0
6,1,0,1,0,0,0,18756,0,0,0,14,22,11,0,5
7,0,3,1,0,0,0,0,2877,2750,0,1192,62,1396,0,0
8,0,1,0,0,0,0,0,1,1973,0,5,0,2,0,0
9,0,0,0,0,0,0,0,0,0,346,0,0,0,0,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========


              precision    recall  f1-score   support

           0       0.74      0.23      0.35     90000
           1       0.93      0.36      0.52     90000
           2       0.80      0.16      0.26     39772
           3       0.68      1.00      0.81     29040
           4       0.97      0.30      0.46     28907
           5       0.71      0.57      0.63     27955
           6       0.73      1.00      0.84     18810
           7       0.11      0.35      0.17      8281
           8       0.09      1.00      0.17      1982
           9       0.18      1.00      0.30       346
          10       0.00      0.86      0.00       111
          11       0.02      0.87      0.03        46
          12       0.00      0.59      0.00        17
          13       1.00      1.00      1.00        11
          14       0.53      0.90      0.67        10

    accuracy                           0.41    335288
   macro avg       0.50      0.68      0.41    335288
weighted avg       0.79   

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"]),
        "roc_auc": float(metricas_test["roc_auc"]),

    }
}

summary_test

{'experimento': 'CIC18__NearMiss_SMOTE__v1__pca4_knn__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC18__NearMiss_SMOTE__v1/CIC18__NearMiss_SMOTE__v1__test.csv',
 'shape_test': {'rows': 335288, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_test': {'accuracy': 0.40879482713368803,
  'precision_weighted': 0.7915511516878581,
  'recall_weighted': 0.40879482713368803,
  'f1_weighted': 0.4799704398015179,
  'precision_macro': 0.500254392175916,
  'recall_macro': 0.6778213100727263,
  'f1_macro': 0.4145650433710809,
  'mcc': 0.3886948959213576,
  'roc_auc': 0.7171753331657125}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1/CIC18__NearMiss_SMOTE__v1__pca4_knn__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.9829533333333333, 'precision_weighted': 0.9829470805148863, 'recall_weighted': 0.9829533333333333, 'f1_weighted': 0.9829205981527336, 'precision_macro': 0.9829470805148863, 'recall_macro': 0.9829533333333333, 'f1_macro': 0.9829205981527336, 'mcc': 0.981740108031443, 'roc_auc': 0.9969674888095239, 'fit_time': 0.20621604919433595, 'score_time': 0.2991933345794678}

TEST:
{'accuracy': 0.40879482713368803, 'precision_weighted': 0.7915511516878581, 'recall_weighted': 0.40879482713368803, 'f1_weighted': 0.4799704398015179, 'precision_macro': 0.500254392175916, 'recall_macro': 0.6778213100727263, 'f1_macro': 0.4145650433710809, 'mcc': 0.3886948959213576, 'roc_auc': 0.7171753331657125}
